In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q07/annotated/checkpoints/pre_cell_5.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 5 ###



lineitem_filtered = lineitem[
    (lineitem["L_SHIPDATE"] >= pd.Timestamp("1995-01-01"))
    & (lineitem["L_SHIPDATE"] < pd.Timestamp("1997-01-01"))
]
lineitem_filtered["L_YEAR"] = lineitem_filtered["L_SHIPDATE"].dt.year
lineitem_filtered["VOLUME"] = lineitem_filtered["L_EXTENDEDPRICE"] * (
    1.0 - lineitem_filtered["L_DISCOUNT"]
)
lineitem_filtered = lineitem_filtered.loc[
    :, ["L_ORDERKEY", "L_SUPPKEY", "L_YEAR", "VOLUME"]
]
supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NATIONKEY"]]
orders_filtered = orders.loc[:, ["O_ORDERKEY", "O_CUSTKEY"]]
customer_filtered = customer.loc[:, ["C_CUSTKEY", "C_NATIONKEY"]]
n1 = nation[(nation["N_NAME"] == "FRANCE")].loc[:, ["N_NATIONKEY", "N_NAME"]]
n2 = nation[(nation["N_NAME"] == "GERMANY")].loc[:, ["N_NATIONKEY", "N_NAME"]]

# ----- do nation 1 -----
N1_C = customer_filtered.merge(
    n1, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N1_C = N1_C.drop(columns=["C_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "CUST_NATION"}
)
N1_C_O = N1_C.merge(
    orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="inner"
)
N1_C_O = N1_C_O.drop(columns=["C_CUSTKEY", "O_CUSTKEY"])

N2_S = supplier_filtered.merge(
    n2, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N2_S = N2_S.drop(columns=["S_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "SUPP_NATION"}
)
N2_S_L = N2_S.merge(
    lineitem_filtered, left_on="S_SUPPKEY", right_on="L_SUPPKEY", how="inner"
)
N2_S_L = N2_S_L.drop(columns=["S_SUPPKEY", "L_SUPPKEY"])

total1 = N1_C_O.merge(
    N2_S_L, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner"
)
total1 = total1.drop(columns=["O_ORDERKEY", "L_ORDERKEY"])

# ----- do nation 2 -----
N2_C = customer_filtered.merge(
    n2, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N2_C = N2_C.drop(columns=["C_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "CUST_NATION"}
)
N2_C_O = N2_C.merge(
    orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="inner"
)
N2_C_O = N2_C_O.drop(columns=["C_CUSTKEY", "O_CUSTKEY"])

N1_S = supplier_filtered.merge(
    n1, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N1_S = N1_S.drop(columns=["S_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "SUPP_NATION"}
)
N1_S_L = N1_S.merge(
    lineitem_filtered, left_on="S_SUPPKEY", right_on="L_SUPPKEY", how="inner"
)
N1_S_L = N1_S_L.drop(columns=["S_SUPPKEY", "L_SUPPKEY"])

total2 = N2_C_O.merge(
    N1_S_L, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner"
)
total2 = total2.drop(columns=["O_ORDERKEY", "L_ORDERKEY"])

# concat results
total = pd.concat([total1, total2])

total = total.groupby(["SUPP_NATION", "CUST_NATION", "L_YEAR"], as_index=False).agg(
    REVENUE=pd.NamedAgg(column="VOLUME", aggfunc="sum")
)



In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/annotated/checkpoints/post_cell_5.pickle